In [1]:
import os 
import sys
import json
import random
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

load_dotenv(project_root / ".env")


openrouter_key = os.getenv("OPENROUTER_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

if not openrouter_key and not openai_key:
    raise EnvironmentError(
        "❌ No API key found!\n"
        "   Add OPENROUTER_API_KEY (recommended) or OPENAI_API_KEY to .env"
    )

from real_state.config import (
    CRAWL_OUT_DIR, VECTOR_DIR, EMBEDDING_MODEL, PROVIDER
)

random.seed(42)

provider = "openrouter" if openrouter_key else "OpenAI"
print("✅ Environment loaded")
print(f"🌐 Provider: {provider}")
print(f"📁 Project root: {project_root}")

✅ Environment loaded
🌐 Provider: openrouter
📁 Project root: d:\ai\bootcamp\mini-project-02\real_state


## Import chunking Services


In [2]:
from real_state.application.ingest_document_service import (
    sementic_chunk,
    fixed_chunk,
    sliding_chunks,
    parent_child_chunk,
    late_chunk_index,
    late_chunk_split
)

## Load Corpus

In [4]:
jsonl_path = CRAWL_OUT_DIR / "primelands_corpus.jsonl"

if not jsonl_path.exists():
    raise FileExistsError(f"❌ Corpus not found.")

with open(jsonl_path, 'r', encoding='utf-8') as f:
    documents = [json.loads(line) for line in f]

print(f"✅ Loaded {len(documents)} documents")
print(f"📊 Total content size: {sum(len(d['content']) for d in documents):,} chars")

✅ Loaded 206 documents
📊 Total content size: 375,441 chars


## Apply chunking strategies

In [5]:
import shutil

if VECTOR_DIR.exists():
    print(f"🗑️  Removing existing vector store: {VECTOR_DIR}")
    shutil.rmtree(VECTOR_DIR)
    print("   ✅ Cleaned up")

VECTOR_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Fresh vector directory ready: {VECTOR_DIR}")


🗑️  Removing existing vector store: d:\ai\bootcamp\mini-project-02\real_state\data\vector_store
   ✅ Cleaned up
📁 Fresh vector directory ready: d:\ai\bootcamp\mini-project-02\real_state\data\vector_store


In [5]:
print("🔄 Running semantic chunking...")
semantic_chunks = sementic_chunk(documents)


semantic_path = CRAWL_OUT_DIR / "chunks_semantic.jsonl"
with open(semantic_path, 'w', encoding='utf-8') as f:
    for chunk in semantic_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

print(f"✅ Semantic chunking complete: {len(semantic_chunks)} chunks")
print(f"💾 Saved to: {semantic_path}")

🔄 Running semantic chunking...
✅ Semantic chunking complete: 206 chunks
💾 Saved to: d:\ai\bootcamp\mini-project-02\real_state\data\chunks_semantic.jsonl


In [6]:
print("🔄 Running fixed-window chunking...")
fixed_chunks = fixed_chunk(documents)

fixed_path = CRAWL_OUT_DIR / "chunks_fixed.jsonl"
with open(fixed_path, 'w', encoding='utf-8') as f:
    for chunk in fixed_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

avg_tokens = sum(c['token_count'] for c in fixed_chunks) / len(fixed_chunks) if fixed_chunks else 0
print(f"✅ Fixed chunking complete: {len(fixed_chunks)} chunks")
print(f"📊 Avg token count: {avg_tokens:.1f}")
print(f"💾 Saved to: {fixed_path}")

🔄 Running fixed-window chunking...
✅ Fixed chunking complete: 209 chunks
📊 Avg token count: 455.6
💾 Saved to: d:\ai\bootcamp\mini-project-02\real_state\data\chunks_fixed.jsonl


In [8]:
print("🔄 Running sliding-window chunking...")
sliding_chunks = sliding_chunks(documents)

sliding_path = CRAWL_OUT_DIR / "chunks_sliding.jsonl"
with open(sliding_path, 'w', encoding="utf-8") as f:
    for chunk in sliding_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

print(f"✅ Sliding chunking complete: {len(sliding_chunks)} chunks")
print(f"💾 Saved to: {sliding_path}")

🔄 Running sliding-window chunking...
✅ Sliding chunking complete: 474 chunks
💾 Saved to: d:\ai\bootcamp\mini-project-02\real_state\data\chunks_sliding.jsonl


In [7]:
print("🔄 Running parent-child chunking...")
child_chunks, parent_chunks = parent_child_chunk(documents)


parent_path = CRAWL_OUT_DIR / "parent_chunk.jsonl"
child_path = CRAWL_OUT_DIR / "child_chunk.jsonl"

with open(parent_path, "w", encoding="utf-8") as f:
    for chunk in parent_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

with open(child_path, "w", encoding="utf-8") as f:
    for chunk in child_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

print(f"✅ Parent child chunking complete: \nparent len : {len(parent_chunks)}\nchild len: {len(child_chunks)}")
print(f"💾 save to: \nparent path: {parent_path}\nchild chunks: {child_path}")
        

🔄 Running parent-child chunking...
✅ Parent child chunking complete: 
parent len : 206
child len: 531
💾 save to: 
parent path: d:\ai\bootcamp\mini-project-02\real_state\data\parent_chunk.jsonl
child chunks: d:\ai\bootcamp\mini-project-02\real_state\data\child_chunk.jsonl


In [3]:
from real_state.infrastructure.llm_provider import get_default_embeddings

In [4]:
from real_state.infrastructure.db.vector_db import QdrantStorage

In [ ]:
client = QdrantClient()